In [ ]:
! pip install -U wikontic

In [ ]:
from wikontic.utils.structured_inference_with_db import StructuredInferenceWithDB
from wikontic.utils.openai_utils import LLMTripletExtractor
from wikontic.utils.structured_aligner import Aligner

from pymongo.mongo_client import MongoClient

In [ ]:
from langchain.agents import create_agent

from openai import OpenAI, DefaultHttpxClient
from dotenv import load_dotenv, find_dotenv

from langchain_openai import ChatOpenAI
import httpx

import os

## Before we start...

To use Wikontic, you’ll need an API key for OpenAI or OpenRouter to make requests to your preferred LLM.

Store your credentials in a ```.env``` file located in the same directory as this notebook:

- ```BASE_URL``` — the service endpoint

- ```KEY``` — your API key

- ```PROXY_URL``` — proxy url, if you need one; otherwise set it to None in the next notebook cell

In [ ]:
_ = load_dotenv(find_dotenv())

base_url = os.getenv("OPENROUTER_BASE_URL")
api_key = os.getenv("KEY")
proxy_url = os.getenv("PROXY_URL")
http_client = httpx.Client(proxy=proxy_url)

## Extract Triplets

Using the ```LLMTripletExtractor``` class, you can extract factual triplets in the form (subject, relation, object).

The extracted structure follows the Wikidata format:

- subject — subject entity

- relation — relation connecting the subject and object

- object — object entity

- qualifiers — a list of dictionaries, where each dictionary contains:

    - relation — relation between the main triplet and the qualifier object

    - object — qualifier entity connected to the main triplet

- subject_type — class describing the subject entity

- object_type — class describing the object entity

In [ ]:
from wikontic.utils.openai_utils import LLMTripletExtractor

extractor = LLMTripletExtractor(model="gpt-4o", api_key=api_key, proxy=proxy_url)

In [ ]:
text_1 = """
    Steven Paul Jobs (February 24, 1955 – October 5, 2011) was an American businessman, inventor, 
    and investor best known for co-founding the technology company Apple Inc.
    """

In [ ]:
triplets = extractor.extract_triplets_from_text(text=text_1)
triplets

## Add database storage

To use Wikontic, you'll need to create a database to store the triplets.

To start a mongoDB container, run:

```

docker pull mongodb/mongodb-atlas-local
docker run -d --name wikontic -p 27018:27017 mongodb/mongodb-atlas-local:latest

```

## Non-ontological KG construction pipeline 

Id you do not need to use ontology, you can use the pipeline with entity/relations name refinement as follows:

In [ ]:
from wikontic.create_triplets_db import create_triplets_database

create_triplets_database(
    mongo_uri="mongodb://localhost:27018/?directConnection=true",
    db_name="test_db_non_onto",
)

In [ ]:
from wikontic.utils.inference_with_db import InferenceWithDB
from wikontic.utils.dynamic_aligner import Aligner


def get_mongo_client(mongo_uri):
    client = MongoClient(mongo_uri)
    return client

In [ ]:
mongo_client = get_mongo_client("mongodb://localhost:27018/?directConnection=true")
triplets_db = mongo_client.get_database("test_db_non_onto")
triplets_db.list_collection_names()
coll = triplets_db.get_collection("triplets")

In [ ]:
mongo_client = get_mongo_client("mongodb://localhost:27018/?directConnection=true")
triplets_db = mongo_client.get_database("test_db_non_onto")

extractor = LLMTripletExtractor(model="gpt-4o", api_key=api_key, proxy=proxy_url)

aligner = Aligner(triplets_db=triplets_db)
inferer = InferenceWithDB(extractor, aligner, triplets_db)

In [ ]:
triplets = inferer.extract_triplets_and_add_to_db(text=text_1, source_text_id="1")
triplets

Further, you can query the resulted KG to answer questions.

In [ ]:
ents = inferer.identify_relevant_entities_from_question_with_llm(
    "Who was Jobs?", use_entity_types=False
)

supporting_triplets, answer = inferer.answer_question_with_llm("Who was Jobs?", ents)
answer

## Non-ontological KG pipeline with langchain

You can use functions of different variations of Wikontic's pipeline as langchain tools for your agents:

In [ ]:
model = ChatOpenAI(
    model="gpt-4o", base_url=base_url, api_key=api_key, http_client=http_client
)

langchain_agent = create_agent(
    model,
    tools=[
        inferer.extract_triplets_and_add_to_db_tool,
        inferer.retrieve_similar_entity_names_tool,
        inferer.get_1_hop_supporting_triplets_tool,
        inferer.answer_question_with_llm_tool,
    ],
)

In [ ]:
text_2 = """Jobs was also the founder of NeXT and chairman and majority shareholder of Pixar.
        He was a pioneer of the personal computer revolution of the 1970s and 1980s,
        along with his early business partner and fellow Apple co-founder Steve Wozniak."""

langchain_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": text_2,
            }
        ]
    }
)

In [ ]:
langchain_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What are the companies Jobs founded?",
            }
        ]
    }
)

----

## Use full Wikontic's capabilities - Wikidata Ontology + name refinement

In [ ]:
from wikontic.create_wikidata_ontology_db import create_wikidata_ontology_database
from wikontic.create_ontological_triplets_db import (
    create_ontological_triplets_database,
)

In [ ]:
create_wikidata_ontology_database(
    mongo_uri="mongodb://localhost:27018/?directConnection=true",
    database="wikidata_ontology_test",
)

create_ontological_triplets_database(
    mongo_uri="mongodb://localhost:27018/?directConnection=true",
    db_name="test_db_onto",
)

In [ ]:
from wikontic.utils.structured_inference_with_db import StructuredInferenceWithDB
from wikontic.utils.structured_aligner import Aligner

triplets_db = mongo_client.get_database("test_db_onto")
ontology_db = mongo_client.get_database("wikidata_ontology_test")

aligner = Aligner(triplets_db=triplets_db, ontology_db=ontology_db)
inferer = StructuredInferenceWithDB(extractor, aligner, triplets_db)

In [ ]:
triplets = inferer.extract_triplets_with_ontology_filtering_and_add_to_db(
    text=text_1, sample_id="1"
)
triplets

In [ ]:
ents = inferer.identify_relevant_entities_from_question_with_llm(
    "When the person that founded Apple was born?"
)
supporting_triplets, answer = inferer.answer_question_with_llm(
    "When the person that founded Apple was born?", linked_entities=ents
)
answer

## Use full Wikontic's capabilities with langchain in an agentic pipeine!

In [ ]:
model = ChatOpenAI(
    model="gpt-4o", base_url=base_url, api_key=api_key, http_client=http_client
)

langchain_agent = create_agent(
    model,
    tools=[
        inferer.extract_triplets_with_ontology_filtering_and_add_to_db_tool,
        inferer.retrieve_similar_entity_names_tool,
        inferer.get_1_hop_supporting_triplets_tool,
        inferer.answer_question_with_llm_tool,
    ],
)

In [ ]:
langchain_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": text_2,
            }
        ]
    }
)

In [ ]:
langchain_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What are the companies Jobs founded?",
            }
        ]
    }
)